# SARIMA — Walmart Store Sales Forecasting

## Goal
Forecast weekly `Weekly_Sales` with classical seasonal ARIMA. Metric = **WMAE** (holiday weight **5**).

## Why SARIMA (theory)
SARIMA$(p,d,q)\times(P,D,Q)_s$ models:
- short-term autocorrelation (`p`,`q`) after differencing (`d`)
- **yearly** seasonality on weekly data with **$s=52$**
- seasonal AR/MA (`P`,`Q`) after seasonal differencing (`D`)

**How we choose orders (theory → practice):**
1. ADF on train series → need for `d` / `D`
2. ACF/PACF after differencing → candidate `q`/`Q` and `p`/`P`
3. Prefer **simple** seasonal structures (airline-like $(0,1,1)\times(0,1,1)_{52}$) — often generalize better than full $(1,1,1)\times(1,1,1)_{52}$

## Protocol (leakage-safe)
1. Split **first**: last **8 weeks** = validation
2. EDA / ADF / ACF only on **train** (never peek at val)
3. Fit on train; `forecast(steps=len(val))` — no val `y` in fit
4. Screen orders on Store 1 / Dept 1
5. Confirm pruned grid on **Subset50**; champion = min Subset50 WMAE
6. **Submission**: refit champion on full `train.csv`, forecast `test.csv` → `submission_sarima.csv`

`test.csv` never used for fit / selection — only for final submission. Keep the order grid **small** — theory matters more than brute force.


In [ ]:
#1 — setup
# Internet ON. CPU only (no GPU needed). Expect 1–3h for full submission.
!pip install -q statsmodels dagshub mlflow

import os, warnings, logging, json
from pathlib import Path
from contextlib import contextmanager

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose
import mlflow

warnings.filterwarnings('ignore')
logging.getLogger('statsmodels').setLevel(logging.ERROR)

NOTEBOOK_VERSION = 'sarima_v2_clean'
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
WORK_DIR.mkdir(parents=True, exist_ok=True)

USE_MLFLOW = False
try:
    import dagshub
    token = None
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('DAGSHUB_USER_TOKEN')
    except Exception:
        token = os.environ.get('DAGSHUB_USER_TOKEN')

    if token:
        os.environ['DAGSHUB_USER_TOKEN'] = token
        dagshub.auth.add_app_token(token)
        dagshub.init(
            repo_owner='lshek22',
            repo_name='walmart-recruiting-store-sales-forecasting',
            mlflow=True,
        )
        USE_MLFLOW = True
        print('DagsHub/MLflow: token auth OK')
    else:
        print('DagsHub token missing — MLflow disabled (optional; notebook still runs)')
except Exception as e:
    print(f'MLflow disabled ({type(e).__name__}: {e})')

if USE_MLFLOW:
    mlflow.set_experiment('SARIMA_Training')
else:
    @contextmanager
    def _noop_run(*args, **kwargs):
        class _Dummy:
            def __enter__(self): return self
            def __exit__(self, *a): return False
            def __getattr__(self, name):
                return lambda *a, **k: None
        yield _Dummy()
    mlflow.start_run = _noop_run
    mlflow.log_params = lambda *a, **k: None
    mlflow.log_param = lambda *a, **k: None
    mlflow.log_metric = lambda *a, **k: None
    mlflow.log_artifact = lambda *a, **k: None
    mlflow.log_text = lambda *a, **k: None

print(f'Ready | {NOTEBOOK_VERSION} | mlflow={USE_MLFLOW}')


In [ ]:
#2 — load + clean data
COMPETITION_SLUG = 'walmart-recruiting-store-sales-forecasting'


def find_competition_data_dir():
    preferred = f'/kaggle/input/competitions/{COMPETITION_SLUG}'
    if os.path.isdir(preferred):
        return preferred
    for root, _, files in os.walk('/kaggle/input'):
        if {'train.csv', 'train.csv.zip'} & set(files):
            return root
    raise FileNotFoundError(
        'Competition data not found. Kaggle: Add Input → Competitions → '
        'Walmart Recruiting - Store Sales Forecasting'
    )


def read_competition_csv(data_dir, stem):
    for name in (f'{stem}.csv', f'{stem}.csv.zip'):
        path = os.path.join(data_dir, name)
        if os.path.exists(path):
            return pd.read_csv(path)
    raise FileNotFoundError(f'Missing {stem}.csv / .csv.zip in {data_dir}')


DATA_DIR = find_competition_data_dir()
print('DATA_DIR:', DATA_DIR)

train = read_competition_csv(DATA_DIR, 'train')
test = read_competition_csv(DATA_DIR, 'test')
stores = read_competition_csv(DATA_DIR, 'stores')
features = read_competition_csv(DATA_DIR, 'features')

for df in (train, test, features):
    df['Date'] = pd.to_datetime(df['Date'])

train = train.merge(stores, on='Store', how='left')
train = train.merge(features.drop(columns=['IsHoliday']), on=['Store', 'Date'], how='left')
test = test.merge(stores, on='Store', how='left')
test = test.merge(features.drop(columns=['IsHoliday']), on=['Store', 'Date'], how='left')

md_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
train[md_cols] = train[md_cols].fillna(0)
test[md_cols] = test[md_cols].fillna(0)
train['Weekly_Sales'] = train['Weekly_Sales'].clip(lower=0)

train = train.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
test = test.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

print(f'Train {train.shape} | Test {test.shape} | series={train.groupby(["Store","Dept"]).ngroups}')
print('Note: test.csv loaded for schema only — never used for fit / selection.')


In [ ]:
#3 — split FIRST (EDA must not see val weeks)
def wmae(y_true, y_pred, is_holiday):
    w = np.where(np.asarray(is_holiday).astype(bool), 5, 1)
    return float(np.sum(w * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(w))


VAL_WEEKS = 8
SEASONAL_PERIOD = 52
SPLIT_DATE = train['Date'].max() - pd.Timedelta(weeks=VAL_WEEKS)
train_set = train[train['Date'] <= SPLIT_DATE].copy()
val_set = train[train['Date'] > SPLIT_DATE].copy()

print(f'Train: {train_set.Date.min().date()} → {train_set.Date.max().date()}')
print(f'Val  : {val_set.Date.min().date()} → {val_set.Date.max().date()} ({val_set.Date.nunique()} weeks)')

with mlflow.start_run(run_name='SARIMA_Setup'):
    mlflow.log_params({
        'notebook_version': NOTEBOOK_VERSION,
        'val_weeks': VAL_WEEKS,
        'split_date': str(SPLIT_DATE.date()),
        'seasonal_period': SEASONAL_PERIOD,
        'metric': 'WMAE',
        'holiday_weight': 5,
        'leakage_protocol': 'split_before_eda_fit_train_only',
    })


## Theory check on **train only** (Store 1 / Dept 1)

ADF → differencing. ACF/PACF after seasonal+first differencing → MA/AR candidates.  
All plots use `train_set` only.


In [ ]:
#4 — EDA / stationarity on TRAIN only
STORE, DEPT = 1, 1


def get_series(df, store, dept, quiet=False):
    s = (
        df[(df['Store'] == store) & (df['Dept'] == dept)]
        .set_index('Date')['Weekly_Sales']
        .asfreq('W-FRI')
        .astype(float)
    )
    # asfreq can introduce NaNs for missing weeks — interpolate only on known history
    if s.isna().any():
        n_nan = int(s.isna().sum())
        s = s.interpolate(limit_direction='both')
        if not quiet:
            print(f'  interpolated {n_nan} missing week(s) for Store {store} Dept {dept}')
    return s


def adf_report(series, label):
    series = series.dropna()
    if len(series) < 20:
        print(f'{label}: too short for ADF')
        return
    stat, pval, *_ = adfuller(series, autolag='AIC')
    print(f'{label:40s} ADF={stat:8.3f}  p={pval:.4f}  → {"stationary" if pval < 0.05 else "non-stationary"}')


series_train = get_series(train_set, STORE, DEPT)
val_series = get_series(val_set, STORE, DEPT)
print(f'Screening series: Store {STORE} Dept {DEPT} | train={len(series_train)} val={len(val_series)}')

seas = series_train.diff(SEASONAL_PERIOD)
seas_first = seas.diff(1)

print('\nADF (train only):')
adf_report(series_train, 'raw')
adf_report(seas, f'seasonal diff (s={SEASONAL_PERIOD})')
adf_report(seas_first, 'seasonal + first diff')

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
fig.patch.set_facecolor('#0f1117')
for ax in axes.ravel():
    ax.set_facecolor('#1a1d27')
    ax.tick_params(colors='#8b949e')

axes[0, 0].plot(series_train.index, series_train.values, color='#58a6ff', lw=1)
axes[0, 0].set_title('Train series', color='white')

decomp = seasonal_decompose(series_train.dropna(), model='additive', period=SEASONAL_PERIOD)
axes[0, 1].plot(decomp.trend.dropna().index, decomp.trend.dropna().values, color='#f78166', lw=1)
axes[0, 1].set_title('Trend (decompose)', color='white')

x = seas_first.dropna()
nlags = min(60, max(10, len(x) // 3))
acf_vals = acf(x, nlags=nlags, fft=True)
pacf_vals = pacf(x, nlags=min(40, nlags), method='ywm')
axes[1, 0].bar(range(len(acf_vals)), acf_vals, color='#8b949e')
axes[1, 0].set_title('ACF after seas+first diff', color='white')
axes[1, 1].bar(range(len(pacf_vals)), pacf_vals, color='#8b949e')
axes[1, 1].set_title('PACF after seas+first diff', color='white')
plt.tight_layout()
plt.savefig(str(WORK_DIR / 'sarima_eda_train.png'), dpi=130, facecolor='#0f1117')
plt.show()
print('Interpretation hint: significant ACF lag 1 → try q=1; seasonal spike near 52 → Q=1; keep D=1.')


In [ ]:
#5 — helpers + pruned order grid (theory-led, fast)
FIT_KW = dict(
    disp=False,
    method='lbfgs',
    maxiter=80,
)


def fit_forecast_sarima(y_train, y_val, order, seasonal_order, exog_train=None, exog_val=None):
    model = SARIMAX(
        y_train,
        order=order,
        seasonal_order=seasonal_order,
        exog=exog_train,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    result = model.fit(**FIT_KW)
    fc = result.forecast(steps=len(y_val), exog=exog_val)
    preds = np.clip(np.asarray(fc, dtype=float), 0, None)
    return result, preds


# Pruned grid: airline-like + a few simple variants (NOT full factorial)
ORDERS = {
    '011x011': ((0, 1, 1), (0, 1, 1, SEASONAL_PERIOD)),  # classic airline
    '111x011': ((1, 1, 1), (0, 1, 1, SEASONAL_PERIOD)),  # + AR
    '111x010': ((1, 1, 1), (0, 1, 0, SEASONAL_PERIOD)),  # seasonal diff only
}

print('Order grid (pruned for speed + theory):')
for k, (o, so) in ORDERS.items():
    print(f'  {k}: {o} x {so}')


## Stage A — single-series screen

Compare pruned orders on Store 1 / Dept 1, then SARIMAX(+`IsHoliday`) on the best order.


In [ ]:
#6 — single-series order screen
holiday_tr = (
    train_set[(train_set['Store'] == STORE) & (train_set['Dept'] == DEPT)]
    .set_index('Date')['IsHoliday']
    .asfreq('W-FRI')
    .fillna(0)
    .astype(float)
)
holiday_vl = (
    val_set[(val_set['Store'] == STORE) & (val_set['Dept'] == DEPT)]
    .set_index('Date')['IsHoliday']
    .asfreq('W-FRI')
    .fillna(0)
    .astype(float)
)

single_results = {}
print(f'{"Config":<14} {"AIC":>10} {"WMAE":>10}')
print('-' * 38)

for name, (order, seasonal) in ORDERS.items():
    try:
        res, preds = fit_forecast_sarima(series_train, val_series, order, seasonal)
        score = wmae(val_series.values, preds, holiday_vl.reindex(val_series.index).fillna(0).values)
        single_results[name] = {'wmae': score, 'aic': float(res.aic), 'order': order, 'seasonal': seasonal, 'exog': False}
        print(f'{name:<14} {res.aic:>10.1f} {score:>10,.2f}')
    except Exception as e:
        print(f'{name:<14} FAILED ({type(e).__name__}: {e})')

assert single_results, 'All single-series SARIMA fits failed'
best_single = min(single_results, key=lambda k: single_results[k]['wmae'])
best_order = single_results[best_single]['order']
best_seasonal = single_results[best_single]['seasonal']
print(f'\nBest order on S1D1: {best_single} WMAE={single_results[best_single]["wmae"]:,.2f}')

# SARIMAX with IsHoliday on the best order (fair compare)
try:
    res_x, preds_x = fit_forecast_sarima(
        series_train, val_series, best_order, best_seasonal,
        exog_train=holiday_tr.reindex(series_train.index).fillna(0),
        exog_val=holiday_vl.reindex(val_series.index).fillna(0),
    )
    score_x = wmae(val_series.values, preds_x, holiday_vl.reindex(val_series.index).fillna(0).values)
    single_results['SARIMAX_IsHoliday'] = {
        'wmae': score_x, 'aic': float(res_x.aic),
        'order': best_order, 'seasonal': best_seasonal, 'exog': True,
    }
    print(f'{"SARIMAX_IsHoliday":<14} {res_x.aic:>10.1f} {score_x:>10,.2f}')
except Exception as e:
    print(f'SARIMAX_IsHoliday FAILED ({type(e).__name__}: {e})')

with mlflow.start_run(run_name='SARIMA_S1D1_OrderScreen'):
    mlflow.log_param('best_single', best_single)
    for name, info in single_results.items():
        mlflow.log_metric(f'wmae_{name}', info['wmae'])
        mlflow.log_metric(f'aic_{name}', info['aic'])


In [ ]:
#7 — plot best SARIMA vs SARIMAX on S1D1 (fair compare)
res_best, preds_best = fit_forecast_sarima(series_train, val_series, best_order, best_seasonal)
preds_x = None
if 'SARIMAX_IsHoliday' in single_results:
    _, preds_x = fit_forecast_sarima(
        series_train, val_series, best_order, best_seasonal,
        exog_train=holiday_tr.reindex(series_train.index).fillna(0),
        exog_val=holiday_vl.reindex(val_series.index).fillna(0),
    )

fig, axes = plt.subplots(1, 2 if preds_x is not None else 1, figsize=(12, 4), squeeze=False)
fig.patch.set_facecolor('#0f1117')
axes = axes.ravel()
for ax in axes:
    ax.set_facecolor('#1a1d27')
    ax.tick_params(colors='#8b949e')
    ax.plot(val_series.index, val_series.values, color='#58a6ff', label='Actual', lw=2)

axes[0].plot(val_series.index, preds_best, color='#f78166', label=f'SARIMA {best_single}', lw=2)
axes[0].set_title(f'SARIMA {best_single} | WMAE={single_results[best_single]["wmae"]:,.0f}', color='white')
axes[0].legend(facecolor='#1a1d27', labelcolor='white')

if preds_x is not None:
    axes[1].plot(val_series.index, preds_x, color='#f78166', label='SARIMAX+IsHoliday', lw=2)
    axes[1].set_title(
        f'SARIMAX+IsHoliday | WMAE={single_results["SARIMAX_IsHoliday"]["wmae"]:,.0f}',
        color='white',
    )
    axes[1].legend(facecolor='#1a1d27', labelcolor='white')

plt.tight_layout()
plt.savefig(str(WORK_DIR / 'sarima_s1d1_compare.png'), dpi=130, facecolor='#0f1117')
plt.show()


## Stage B — Subset50 confirmation

Evaluate **all pruned orders** (+ SARIMAX on best single-series order) on 50 series.

Holdout is **Sept–Oct** (few holidays). For **submission** we prefer **SARIMAX + IsHoliday** when it fits, because Kaggle test + WMAE are holiday-heavy.


In [ ]:
#8 — Subset50
N_SUBSET = 50
MIN_TRAIN_WEEKS = 80


def evaluate_sarima_subset(order, seasonal, use_exog=False, n_series=N_SUBSET, seed=42):
    rng = np.random.default_rng(seed)
    lengths = train_set.groupby(['Store', 'Dept']).size()
    valid_pairs = lengths[lengths >= MIN_TRAIN_WEEKS].index.tolist()
    n = min(n_series, len(valid_pairs))
    sample = [valid_pairs[i] for i in rng.choice(len(valid_pairs), size=n, replace=False)]

    rows, n_failed = [], 0
    fail_reasons = {}
    for store_i, dept_i in sample:
        try:
            tr = get_series(train_set, store_i, dept_i)
            vl = get_series(val_set, store_i, dept_i)
            if len(tr) < MIN_TRAIN_WEEKS or len(vl) == 0:
                n_failed += 1
                fail_reasons['short'] = fail_reasons.get('short', 0) + 1
                continue
            ex_tr = ex_vl = None
            hol_vl = (
                val_set[(val_set['Store'] == store_i) & (val_set['Dept'] == dept_i)]
                .set_index('Date')['IsHoliday']
                .asfreq('W-FRI')
                .fillna(0)
                .astype(float)
            )
            if use_exog:
                hol_tr = (
                    train_set[(train_set['Store'] == store_i) & (train_set['Dept'] == dept_i)]
                    .set_index('Date')['IsHoliday']
                    .asfreq('W-FRI')
                    .fillna(0)
                    .astype(float)
                )
                ex_tr = hol_tr.reindex(tr.index).fillna(0)
                ex_vl = hol_vl.reindex(vl.index).fillna(0)
            _, preds = fit_forecast_sarima(tr, vl, order, seasonal, exog_train=ex_tr, exog_val=ex_vl)
            for i, (dt, actual) in enumerate(vl.items()):
                if i < len(preds):
                    rows.append({
                        'Actual': float(actual),
                        'Predicted': float(preds[i]),
                        'IsHoliday': float(hol_vl.reindex([dt]).fillna(0).iloc[0]),
                    })
        except Exception as e:
            n_failed += 1
            key = type(e).__name__
            fail_reasons[key] = fail_reasons.get(key, 0) + 1

    if not rows:
        return None, n_failed, fail_reasons
    pred_df = pd.DataFrame(rows)
    return wmae(pred_df['Actual'], pred_df['Predicted'], pred_df['IsHoliday']), n_failed, fail_reasons


subset_results = {}
print(f'{"Config":<20} {"Subset50 WMAE":>14} {"failed":>8}')
print('-' * 46)

for name, (order, seasonal) in ORDERS.items():
    score, n_failed, reasons = evaluate_sarima_subset(order, seasonal, use_exog=False)
    subset_results[name] = score
    score_str = f'{score:,.2f}' if score is not None else 'FAIL'
    print(f'{name:<20} {score_str:>14} {n_failed:>8}  {reasons}')
    with mlflow.start_run(run_name=f'SARIMA_Subset50_{name}'):
        mlflow.log_params({'order': order, 'seasonal_order': seasonal, 'exog': False, 'scope': 'subset50'})
        if score is not None:
            mlflow.log_metric('val_wmae_subset50', score)
        mlflow.log_metric('n_failed', n_failed)

# SARIMAX on best single-series order
score_x, n_failed_x, reasons_x = evaluate_sarima_subset(best_order, best_seasonal, use_exog=True)
subset_results['SARIMAX_IsHoliday'] = score_x
score_str = f'{score_x:,.2f}' if score_x is not None else 'FAIL'
print(f'{"SARIMAX_IsHoliday":<20} {score_str:>14} {n_failed_x:>8}  {reasons_x}')
with mlflow.start_run(run_name='SARIMA_Subset50_SARIMAX_IsHoliday'):
    mlflow.log_params({'order': best_order, 'seasonal_order': best_seasonal, 'exog': 'IsHoliday', 'scope': 'subset50'})
    if score_x is not None:
        mlflow.log_metric('val_wmae_subset50', score_x)
    mlflow.log_metric('n_failed', n_failed_x)

valid_subset = {k: v for k, v in subset_results.items() if v is not None}
assert valid_subset, 'All Subset50 SARIMA runs failed'

best_holdout_name = min(valid_subset, key=valid_subset.get)

# Prefer holiday-aware SARIMAX for submission when available
# (Kaggle test + WMAE are holiday-heavy; Sept–Oct holdout underweights that)
if subset_results.get('SARIMAX_IsHoliday') is not None:
    champion_name = 'SARIMAX_IsHoliday'
    print(
        f'Holdout-best (Sept–Oct): {best_holdout_name} = {valid_subset[best_holdout_name]:,.2f}\n'
        f'Submission champion (holiday-aware): {champion_name} = {valid_subset[champion_name]:,.2f}'
    )
else:
    champion_name = best_holdout_name
    print(f'Champion (by Subset50): {champion_name} | WMAE={valid_subset[champion_name]:,.2f}')

champion_subset_wmae = valid_subset[champion_name]


In [ ]:
#9 — chart + save best config
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d27')
names = list(valid_subset.keys())
scores = [valid_subset[k] for k in names]
colors = ['#8b949e'] * len(names)
colors[int(np.argmin(scores))] = '#f78166'
bars = ax.bar(names, scores, color=colors, width=0.55)
ax.bar_label(bars, labels=[f'{v:,.1f}' for v in scores], color='white', fontsize=9)
ax.set_title('SARIMA Subset50 WMAE (lower is better) — champion selection', color='white')
ax.tick_params(colors='#8b949e', axis='x', rotation=15)
plt.tight_layout()
chart_path = WORK_DIR / 'sarima_subset50.png'
plt.savefig(str(chart_path), dpi=130, facecolor='#0f1117')
plt.show()

if champion_name == 'SARIMAX_IsHoliday':
    champ_order, champ_seasonal, champ_exog = best_order, best_seasonal, True
else:
    champ_order, champ_seasonal = ORDERS[champion_name]
    champ_exog = False

spec = {
    'notebook_version': NOTEBOOK_VERSION,
    'champion_config': champion_name,
    'order': list(champ_order),
    'seasonal_order': list(champ_seasonal),
    'exog': 'IsHoliday' if champ_exog else None,
    'selection_criterion': 'holiday_aware_prefer_SARIMAX_IsHoliday',
    'holdout_best_config': best_holdout_name,
    'subset50_wmae': float(champion_subset_wmae),
    'single_series_screen': {k: {'wmae': float(v['wmae']), 'aic': float(v['aic'])} for k, v in single_results.items()},
    'subset50_scores': {k: (float(v) if v is not None else None) for k, v in subset_results.items()},
    'note': 'SARIMA is fit per Store-Dept; artifact is config + theory-led protocol, not a global model.',
}
spec_path = WORK_DIR / 'sarima_best_pipeline_spec.json'
with open(spec_path, 'w', encoding='utf-8') as f:
    json.dump(spec, f, indent=2)

with mlflow.start_run(run_name='SARIMA_Best'):
    mlflow.log_params({
        'champion_config': champion_name,
        'order': champ_order,
        'seasonal_order': champ_seasonal,
        'exog': 'IsHoliday' if champ_exog else None,
        'selection_criterion': 'subset50_val_wmae',
    })
    mlflow.log_metric('val_wmae_subset50', champion_subset_wmae)
    if champion_name in single_results:
        mlflow.log_metric('val_wmae_s1d1', single_results[champion_name]['wmae'])
    mlflow.log_artifact(str(spec_path), 'pipeline')
    mlflow.log_artifact(str(chart_path), 'plots')
    mlflow.log_artifact(str(WORK_DIR / 'sarima_eda_train.png'), 'plots')
    mlflow.log_artifact(str(WORK_DIR / 'sarima_s1d1_compare.png'), 'plots')

print(json.dumps(spec, indent=2))
print(f'Champion ready: {champion_name} | Subset50 WMAE={champion_subset_wmae:,.2f}')
print('Next cell builds Kaggle submission.csv (refit on FULL train history).')


## Stage C — Kaggle submission

Refit the **Subset50 champion** on the **full** `train.csv` history for every `(Store, Dept)` in `test.csv`, then forecast forward to cover test dates.

- Format: `Id = Store_Dept_YYYY-MM-DD`, `Weekly_Sales`
- Failures / short history → last-4-week mean (else global median)
- SARIMA with `s=52` is slow — expect **1–3 hours** for a full run on CPU

Set `SUBMISSION_MAX_SERIES = 50` for a smoke test; `None` for the real file.

In [ ]:
#10 — build submission.csv (SARIMA champion)
MAKE_SUBMISSION = True
SUBMISSION_MAX_SERIES = None  # None = all test series | 50 = quick smoke test

GLOBAL_MEDIAN = float(train['Weekly_Sales'].median())


def fallback_pred(hist_y, n):
    if len(hist_y) >= 1:
        base = float(np.mean(hist_y[-min(4, len(hist_y)):]))
    else:
        base = GLOBAL_MEDIAN
    return np.full(n, max(base, 0.0), dtype=float)


def get_holiday_series(df, store_i, dept_i, index_like):
    hol = (
        df[(df['Store'] == store_i) & (df['Dept'] == dept_i)]
        .set_index('Date')['IsHoliday']
        .asfreq('W-FRI')
        .fillna(0)
        .astype(float)
    )
    return hol.reindex(index_like).fillna(0)


def predict_sarima_test_series(store_i, dept_i, order, seasonal, use_exog):
    te = test[(test['Store'] == store_i) & (test['Dept'] == dept_i)].copy().sort_values('Date')
    if len(te) == 0:
        return []

    hist = get_series(train, store_i, dept_i, quiet=True)  # FULL history
    test_dates = pd.DatetimeIndex(pd.to_datetime(te['Date'].values))

    if len(hist.dropna()) < 20:
        preds = fallback_pred(hist.dropna().values, len(te))
        return list(zip(test_dates, preds, [True] * len(te)))

    try:
        # Build a weekly forecast index from week after last train date through last test date
        start = hist.index.max() + pd.Timedelta(weeks=1)
        end = max(test_dates.max(), start)
        future_idx = pd.date_range(start=start, end=end, freq='W-FRI')
        if len(future_idx) == 0:
            preds = fallback_pred(hist.dropna().values, len(te))
            return list(zip(test_dates, preds, [True] * len(te)))

        ex_tr = ex_fu = None
        if use_exog:
            ex_tr = get_holiday_series(train, store_i, dept_i, hist.index)
            # holidays on future weeks: prefer test IsHoliday, else 0
            hol_te = (
                te.set_index('Date')['IsHoliday']
                .astype(float)
                .reindex(future_idx)
                .fillna(0)
            )
            ex_fu = hol_te

        model = SARIMAX(
            hist,
            order=order,
            seasonal_order=seasonal,
            exog=ex_tr,
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        result = model.fit(**FIT_KW)
        fc = result.forecast(steps=len(future_idx), exog=ex_fu)
        fc = pd.Series(np.clip(np.asarray(fc, dtype=float), 0, None), index=future_idx)

        preds = []
        used_fb = []
        for dt in test_dates:
            if dt in fc.index:
                preds.append(float(fc.loc[dt]))
                used_fb.append(False)
            else:
                # nearest previous forecast or fallback
                prev = fc[fc.index <= dt]
                if len(prev):
                    preds.append(float(prev.iloc[-1]))
                    used_fb.append(False)
                else:
                    preds.append(float(fallback_pred(hist.dropna().values, 1)[0]))
                    used_fb.append(True)
        return list(zip(test_dates, preds, used_fb))
    except Exception:
        preds = fallback_pred(hist.dropna().values, len(te))
        return list(zip(test_dates, preds, [True] * len(te)))


if MAKE_SUBMISSION:
    pairs = test.groupby(['Store', 'Dept']).size().index.tolist()
    if SUBMISSION_MAX_SERIES is not None:
        pairs = pairs[: int(SUBMISSION_MAX_SERIES)]
        print(f'SMOKE TEST: forecasting {len(pairs)} series only', flush=True)
    else:
        print(f'FULL submission: forecasting {len(pairs)} series with config={champion_name}', flush=True)
        print('Note: SARIMA(s=52) is slow — first progress line may take a while.', flush=True)

    rows = []
    n_fallback = 0
    for i, (store_i, dept_i) in enumerate(pairs, 1):
        out = predict_sarima_test_series(
            store_i, dept_i, champ_order, champ_seasonal, champ_exog,
        )
        for dt, pred, used_fb in out:
            rows.append({
                'Id': f"{store_i}_{dept_i}_{pd.Timestamp(dt).strftime('%Y-%m-%d')}",
                'Weekly_Sales': float(pred),
            })
            n_fallback += int(used_fb)
        # print often + flush so Kaggle UI doesn't look frozen
        if i == 1 or i % 10 == 0 or i == len(pairs):
            print(f'  {i}/{len(pairs)} series done | fallback_rows={n_fallback}', flush=True)

    submission = pd.DataFrame(rows)
    if SUBMISSION_MAX_SERIES is None:
        needed = (
            test['Store'].astype(str) + '_' + test['Dept'].astype(str) + '_' + test['Date'].dt.strftime('%Y-%m-%d')
        )
        got = set(submission['Id'])
        missing = [i for i in needed if i not in got]
        if missing:
            print(f'Filling {len(missing)} missing Ids with global median={GLOBAL_MEDIAN:,.2f}')
            submission = pd.concat([
                submission,
                pd.DataFrame({'Id': missing, 'Weekly_Sales': GLOBAL_MEDIAN}),
            ], ignore_index=True)
        order_df = pd.DataFrame({'Id': needed.values})
        submission = order_df.merge(submission, on='Id', how='left')

    sub_path = WORK_DIR / 'submission_sarima.csv'
    # Kaggle rejects blank Weekly_Sales (NaN/inf from rare failed forecasts or merge)
    submission['Weekly_Sales'] = pd.to_numeric(submission['Weekly_Sales'], errors='coerce')
    n_bad = int(submission['Weekly_Sales'].isna().sum() + np.isinf(submission['Weekly_Sales'].to_numpy(dtype=float, copy=False)).sum())
    submission['Weekly_Sales'] = (
        submission['Weekly_Sales']
        .replace([np.inf, -np.inf], np.nan)
        .fillna(GLOBAL_MEDIAN)
        .clip(lower=0)
    )
    assert submission['Weekly_Sales'].notna().all(), 'Weekly_Sales still has nulls'
    assert len(submission) == len(test) or SUBMISSION_MAX_SERIES is not None, (
        f'row count {len(submission)} != test {len(test)}'
    )

    submission.to_csv(sub_path, index=False)
    print(f'Saved {sub_path} | rows={len(submission)} | fallback_rows={n_fallback} | filled_nan_inf={n_bad}')
    print(submission.head())
    print('Weekly_Sales nulls:', int(submission['Weekly_Sales'].isna().sum()))

    with mlflow.start_run(run_name='SARIMA_Submission'):
        mlflow.log_params({
            'champion_config': champion_name,
            'order': champ_order,
            'seasonal_order': champ_seasonal,
            'exog': 'IsHoliday' if champ_exog else None,
            'n_series': len(pairs),
            'smoke_test': SUBMISSION_MAX_SERIES is not None,
            'fallback': 'last4_mean_or_global_median',
        })
        mlflow.log_metric('n_submission_rows', len(submission))
        mlflow.log_metric('n_fallback_rows', n_fallback)
        mlflow.log_artifact(str(sub_path), 'submission')

    print('\\nUpload submission_sarima.csv to Kaggle → Submit Predictions')
    if SUBMISSION_MAX_SERIES is not None:
        print('WARNING: smoke test only — set SUBMISSION_MAX_SERIES = None for a real submission.')
else:
    print('MAKE_SUBMISSION=False — skipped')


In [ ]:
#11 — repair submission CSV if Kaggle rejected blank Weekly_Sales (no need to refit)
# Run this if submission_sarima.csv already exists in /kaggle/working
from pathlib import Path
import numpy as np
import pandas as pd

path = Path('/kaggle/working/submission_sarima.csv')
if not path.exists():
    path = Path('submission_sarima.csv')

sub = pd.read_csv(path)
print('Before:', len(sub), 'null/blank sales:', sub['Weekly_Sales'].isna().sum())

fill = float(train['Weekly_Sales'].median()) if 'train' in dir() else float(sub['Weekly_Sales'].median())
sub['Weekly_Sales'] = pd.to_numeric(sub['Weekly_Sales'], errors='coerce')
sub['Weekly_Sales'] = (
    sub['Weekly_Sales']
    .replace([np.inf, -np.inf], np.nan)
    .fillna(fill)
    .clip(lower=0)
)

out = path.with_name('submission_sarima_fixed.csv')
sub.to_csv(out, index=False)
print('After nulls:', int(sub['Weekly_Sales'].isna().sum()), '| fill=', fill)
print('Saved', out)
print(sub.tail())
from IPython.display import FileLink
display(FileLink(str(out.name)))
